# LifeMate (TripleTrouble): timing notebook

Same code as `/code/predict.sh`: ASR (PhoWhisper-large fine-tuned) -> ReAct agent (Qwen3-4B fine-tuned) -> `env.call` in one `ToolEnv` per sample. Writes `/result/jupyter_submission.jsonl` with a `time` field (ms per sample, from before the sample's ToolEnv and audio to its last call). Fully offline.

## 1. Fixed seed

In [ ]:
import json
import os
import tempfile
import time
from pathlib import Path

import torch  # imported first so seed_everything seeds it too

from lifemate.utils import seed_everything

SEED = 0
seed_everything(SEED)

## 2. Load the models (offline, from the image)

In [ ]:
from lifemate import docker_entry

DATA = Path(os.environ.get("LIFEMATE_DATA", "/data"))        # holds the split folder, e.g. /data/private_test
RESULT = Path(os.environ.get("LIFEMATE_RESULT", "/result"))
CONFIG = Path(os.environ.get("LIFEMATE_SHIPPED", docker_entry.CONFIG_PATH))

config = docker_entry.load_config(CONFIG)
docker_entry.set_offline(config)
data_root, SPLIT = docker_entry.prepare(DATA, Path(tempfile.gettempdir()) / "lifemate_notebook",
                                        docker_entry.REPO_ROOT / "data" / "train")
t_load = time.perf_counter()
pipeline = docker_entry.load_shipped(config, SPLIT, data_root)
print(f"load model: {time.perf_counter() - t_load:.1f} s ({config['agent_preset']} + {config['asr_preset']})")
seed_everything(SEED)  # again, now that the models are built

## 3. Read the test cases

In [ ]:
from lifemate.data import load_samples

test_cases = load_samples(data_root, SPLIT)
print(f"{len(test_cases)} test cases in {SPLIT}")

## 4. Predict and write `/result/jupyter_submission.jsonl`

In [ ]:
from lifemate.infer import run_sample

all_result = []
for item in test_cases:
    line, trace = run_sample(pipeline, item)  # its own ToolEnv; line = id, tool_calls, state_hash, time (ms)
    if trace["error"]:
        print(item["id"], "error:", trace["error"].splitlines()[-1])
    all_result.append(line)

RESULT.mkdir(parents=True, exist_ok=True)
with open(RESULT / "jupyter_submission.jsonl", "w", encoding="utf-8") as f:
    for r in all_result:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
times = [r["time"] for r in all_result]
print(f"predict: {len(all_result)} samples, mean {sum(times) / max(len(times), 1):.0f} ms, max {max(times, default=0)} ms"
      f" -> {RESULT / 'jupyter_submission.jsonl'}")